# CPU latency and memory footprint (Sections 3.8 / 4.5)

Single-session, single-thread CPU latency for every detector configuration: RF-Stat (source / modified), XGBoost-only, LSTM (source), BiLSTM-15s (modified LSTM) and BERT (source / modified), at 15 and 25 m/s.

Fixes three problems of the earlier latency measurement:
1. BiLSTM was timed through Keras `model.predict()`, which adds tens of ms per call. Here the model is called directly (`tf.function`); `predict()` is timed only as a diagnostic row.
2. Thread counts were unequal (XGBoost 1 thread, TF all cores). Here every framework is pinned to 1 thread.
3. Only two configurations were measured. Here all configurations run in ONE session.

Protocol: 100 warm-up passes discarded, 1,000 timed passes, batch size 1. `pure` = forward pass / predict only on pre-processed input; `e2e` = raw window -> features / padding / scaling / tokenisation -> prediction. The whole sequence is repeated `repeats` times (default 3) in the same configuration order. Model size = parameters x 4 bytes (neural nets) or pickle size (RF, XGBoost). `rss_delta_mb` = growth of process resident memory during the timed loop.

**How to run:** use a FRESH Colab runtime (CPU runtime recommended). Run cell 1 first, because thread settings and GPU hiding must be applied before TensorFlow/PyTorch are touched by `pcmef_fixed`. Do not run this notebook after other notebooks in the same session. Weights are untrained: latency does not depend on weight values.

Output: `Detector_Results/tahap_lanjutan/latency_suite.json` (saved after every measurement).

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'TF_NUM_INTRAOP_THREADS', 'TF_NUM_INTEROP_THREADS'):
    os.environ[_v] = '1'

import gc, json, time, pickle, platform
import numpy as np
import psutil
import tensorflow as tf
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError as e:
    raise RuntimeError('TensorFlow sudah diinisialisasi. Restart runtime, lalu jalankan '
                       'cell ini SEBELUM mengimpor pcmef_fixed.') from e
import torch
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

N_WARMUP, N_TIMED = 100, 1000
BERT_MAX_LEN_MODIFIED = {15: 288, 25: 128}   # modified BERT: MAX_LEN per regime (as in the factorial runs)
HF_CACHE = None   # set in the setup cell to a Drive folder so HuggingFace models are downloaded only once

In [ ]:
# ------------------------------------------------------------------ util
def _rss_mb():
    return psutil.Process().memory_info().rss / 2 ** 20


def _stats(t):
    a = np.asarray(t)
    return dict(mean=float(a.mean()), std=float(a.std(ddof=1)), p50=float(np.percentile(a, 50)),
                p95=float(np.percentile(a, 95)), p99=float(np.percentile(a, 99)))


def bench(fn, n_warmup=N_WARMUP, n_timed=N_TIMED):
    for _ in range(n_warmup):
        fn()
    gc.collect()
    rss0 = _rss_mb()
    t = []
    for _ in range(n_timed):
        s = time.perf_counter(); fn(); t.append((time.perf_counter() - s) * 1000)
    out = _stats(t)
    out['rss_delta_mb'] = float(_rss_mb() - rss0)
    out['n_timed'] = n_timed
    return out


def host_info():
    info = dict(python=platform.python_version(), tensorflow=tf.__version__, torch=torch.__version__,
                logical_cpus=os.cpu_count(), ram_gb=round(psutil.virtual_memory().total / 2 ** 30, 1),
                threads='1 (TF intra/inter, torch, OMP/MKL, RF n_jobs, XGB n_jobs)',
                gpu_visible=tf.config.list_physical_devices('GPU'))
    try:
        with open('/proc/cpuinfo') as f:
            info['cpu_model'] = next(l.split(':', 1)[1].strip() for l in f if l.startswith('model name'))
    except Exception:
        info['cpu_model'] = platform.processor()
    for mod in ('transformers', 'sklearn', 'xgboost'):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    return info

In [ ]:
# ------------------------------------------------------------------ data
def _prepare(df, P, seed=42, window_sec=15):
    tr, va, te = P.split_by_episode(df, seed=seed)
    Xtr_l, ytr, _ = P.create_windows_keyed(tr, window_sec, 1)
    Xva_l, yva, _ = P.create_windows_keyed(va, window_sec, 1)
    Xte_l, yte, _ = P.create_windows_keyed(te, window_sec, 1)
    return dict(tr=tr, va=va, te=te, Xtr_l=Xtr_l, ytr=ytr, Xva_l=Xva_l, yva=yva,
                Xte_l=Xte_l, win=Xte_l[0], seed=seed)


# ------------------------------------------------------------------ konfigurasi
def cfg_rf_source(d, P):
    Xtr, ytr, _ = P._rf_source_units(d['tr'])
    sc = StandardScaler().fit(Xtr)
    rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                min_samples_leaf=2, n_jobs=1, random_state=d['seed'],
                                class_weight='balanced').fit(sc.transform(Xtr), ytr)
    te = d['te']
    g = te[te['pseudo_run_id'] == te['pseudo_run_id'].iloc[0]].sort_values('Time')
    block = g[['SNR', 'RSSI', 'PDR', 'Relative_Speed']].values[:5].astype(float)

    def feats():   # 5 sampel terakhir -> VRS (rolling std, ddof=1 seperti pandas) -> 1 baris
        return np.r_[block[-1], np.std(block[:, 3], ddof=1)][None]
    x_pure = sc.transform(feats())
    return dict(pure=lambda: rf.predict(x_pure),
                e2e=lambda: rf.predict(sc.transform(feats())),
                size_mb=len(pickle.dumps(rf)) / 2 ** 20,
                params=int(sum(t.tree_.node_count for t in rf.estimators_)), params_unit='tree nodes')


def cfg_rf_modified(d, P):
    Xtr = P.extract_tabular_features(d['Xtr_l'])
    sc = StandardScaler().fit(Xtr)
    rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                min_samples_leaf=2, n_jobs=1, random_state=d['seed'],
                                class_weight='balanced').fit(sc.transform(Xtr), d['ytr'])
    win = d['win']
    x_pure = sc.transform(P.extract_tabular_features([win]))
    return dict(pure=lambda: rf.predict(x_pure),
                e2e=lambda: rf.predict(sc.transform(P.extract_tabular_features([win]))),
                size_mb=len(pickle.dumps(rf)) / 2 ** 20,
                params=int(sum(t.tree_.node_count for t in rf.estimators_)), params_unit='tree nodes')


def cfg_xgb_only(d, P):
    Xtr = P.extract_tabular_features(d['Xtr_l'])
    Xva = P.extract_tabular_features(d['Xva_l'])
    sc = StandardScaler().fit(Xtr)
    xgb = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=200, max_depth=5,
                        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        random_state=d['seed'], eval_metric='mlogloss',
                        early_stopping_rounds=20, n_jobs=1)
    xgb.fit(sc.transform(Xtr), d['ytr'], eval_set=[(sc.transform(Xva), d['yva'])], verbose=False)
    win = d['win']
    x_pure = sc.transform(P.extract_tabular_features([win]))
    n_nodes = int(len(xgb.get_booster().trees_to_dataframe()))
    return dict(pure=lambda: xgb.predict(x_pure),
                e2e=lambda: xgb.predict(sc.transform(P.extract_tabular_features([win]))),
                size_mb=len(pickle.dumps(xgb)) / 2 ** 20, params=n_nodes, params_unit='tree nodes')


def _cfg_lstm(d, P, arch, with_predict_diag=False):
    allw = d['Xtr_l'] + d['Xva_l'] + d['Xte_l']
    max_len = max(len(x) for x in allw)
    n_feat = d['win'].shape[1]
    sc = StandardScaler().fit(np.vstack(d['Xtr_l']))

    def pad_scale(x):   # sama dengan pad_and_scale di run_lstm_cell
        out = np.zeros((1, max_len, n_feat), dtype=np.float32)
        sl = min(len(x), max_len)
        out[0, :sl] = sc.transform(x[:sl])
        return out
    model = P.build_lstm(arch, (max_len, n_feat), seed=d['seed'])
    f = tf.function(lambda x: model(x, training=False))
    xt = tf.constant(pad_scale(d['win']))
    win = d['win']
    cfg = dict(pure=lambda: f(xt).numpy(),
               e2e=lambda: f(tf.constant(pad_scale(win))).numpy(),
               size_mb=model.count_params() * 4 / 2 ** 20,
               params=int(model.count_params()), params_unit='parameters')
    if with_predict_diag:
        x_np = xt.numpy()
        cfg['diag_keras_predict'] = lambda: model.predict(x_np, verbose=0)
    return cfg


def cfg_lstm_source(d, P):
    return _cfg_lstm(d, P, 'source')


def cfg_lstm_modified(d, P):   # = BiLSTM-15s single-branch di Tabel 7 lama
    return _cfg_lstm(d, P, 'modified', with_predict_diag=True)


def _hf_retry(load, *args, tries=8, wait=15, **kw):
    """Load a HuggingFace object, retrying on Hub errors (429/500) and on corrupted cache files.
    After the first failure the next attempts use force_download=True, which replaces a
    truncated/corrupted cached file (e.g. SafetensorError: header too large)."""
    for i in range(1, tries + 1):
        try:
            return load(*args, cache_dir=HF_CACHE, force_download=(i > 1), **kw)
        except Exception as e:
            if i == tries:
                raise
            print(f'  [HF load failed, retry {i}/{tries - 1} in {wait * i}s, force_download] '
                  f'{type(e).__name__}: {str(e).splitlines()[0][:100]}')
            time.sleep(wait * i)


def _cfg_bert(d, P, arch, speed):
    from transformers import (BertTokenizerFast, BertForSequenceClassification,
                              DistilBertTokenizerFast, DistilBertForSequenceClassification)
    win = d['win']
    if arch == 'source':
        tok = _hf_retry(BertTokenizerFast.from_pretrained, 'bert-base-uncased')
        model = _hf_retry(BertForSequenceClassification.from_pretrained, 'bert-base-uncased', num_labels=3)
        lens = [len(tok.encode(P._tokens_source(x))) for x in d['Xtr_l'][:200]]
        max_len = int(min(512, max(lens)))
        to_text = lambda x: P._tokens_source(x)
    else:
        tok = _hf_retry(DistilBertTokenizerFast.from_pretrained, 'distilbert-base-uncased')
        model = _hf_retry(DistilBertForSequenceClassification.from_pretrained, 'distilbert-base-uncased', num_labels=3)
        max_len = BERT_MAX_LEN_MODIFIED[speed]
        fs = P.compute_feat_stats(d['tr'])
        to_text = lambda x: P._tokens_modified(x, fs, n_bins=20)
    model.eval()

    def enc(x):
        return tok(to_text(x), truncation=True, padding='max_length',
                   max_length=max_len, return_tensors='pt')
    e0 = enc(win)

    def pure():
        with torch.inference_mode():
            return model(input_ids=e0['input_ids'], attention_mask=e0['attention_mask']).logits.argmax(-1)

    def e2e():
        e = enc(win)
        with torch.inference_mode():
            return model(input_ids=e['input_ids'], attention_mask=e['attention_mask']).logits.argmax(-1)
    n = int(sum(p.numel() for p in model.parameters()))
    return dict(pure=pure, e2e=e2e, size_mb=n * 4 / 2 ** 20, params=n,
                params_unit='parameters', max_len=max_len)

In [ ]:
CONFIGS = {
    'RF-Stat (source)': lambda d, P, s: cfg_rf_source(d, P),
    'RF-Stat (modified)': lambda d, P, s: cfg_rf_modified(d, P),
    'XGBoost-only': lambda d, P, s: cfg_xgb_only(d, P),
    'LSTM (source)': lambda d, P, s: cfg_lstm_source(d, P),
    'LSTM (modified) = BiLSTM-15s': lambda d, P, s: cfg_lstm_modified(d, P),
    'BERT (source)': lambda d, P, s: _cfg_bert(d, P, 'source', s),
    'BERT (modified)': lambda d, P, s: _cfg_bert(d, P, 'modified', s),
}

In [ ]:
# ------------------------------------------------------------------ runner
def run_all(dfs_by_speed, P, out_path, repeats=3, configs=None, n_warmup=N_WARMUP,
            n_timed=N_TIMED, n_timed_bert=None, seed=42):
    """dfs_by_speed: {15: dfs15, 25: dfs25} (populasi episode layak, seperti di Tahap_Lanjutan).
    n_timed_bert: isi (mis. 200) hanya bila 1.000 pass BERT terlalu lama; WAJIB disebut di naskah."""
    configs = configs or list(CONFIGS)
    info = host_info()
    print('HOST:', json.dumps(info, indent=1))
    assert not info['gpu_visible'], 'GPU masih terlihat -- restart runtime.'
    built = {}
    for sp, df in dfs_by_speed.items():
        d = _prepare(df, P, seed=seed)
        for name in configs:
            print(f'  membangun {name} @ {sp} m/s ...')
            built[(sp, name)] = CONFIGS[name](d, P, sp)
    results = dict(host=info, protocol=dict(n_warmup=n_warmup, n_timed=n_timed,
                                            n_timed_bert=n_timed_bert or n_timed, repeats=repeats),
                   runs=[])
    for r in range(repeats):
        print(f'\n=== ulangan {r + 1}/{repeats} ===')
        for (sp, name), cfg in built.items():
            nt = (n_timed_bert or n_timed) if name.startswith('BERT') else n_timed
            row = dict(repeat=r + 1, speed=sp, config=name, size_mb=cfg['size_mb'],
                       params=cfg['params'], params_unit=cfg['params_unit'],
                       max_len=cfg.get('max_len'))
            for mode in ('pure', 'e2e', 'diag_keras_predict'):
                if mode in cfg:
                    row[mode] = bench(cfg[mode], n_warmup, nt)
            results['runs'].append(row)
            print(f"  {name:30s} {sp} m/s  pure {row['pure']['mean']:9.3f} ms  "
                  f"e2e {row['e2e']['mean']:9.3f} ms"
                  + (f"  [Keras predict() {row['diag_keras_predict']['mean']:.2f} ms]"
                     if 'diag_keras_predict' in row else ''))
            json.dump(results, open(out_path, 'w'), indent=2)   # simpan tiap langkah
    summarize(results)
    return results


def summarize(results):
    import pandas as pd
    rows = []
    for (sp, name), g in pd.DataFrame(results['runs']).groupby(['speed', 'config'], sort=False):
        rec = dict(speed=sp, config=name, params=g['params'].iloc[0], size_mb=round(g['size_mb'].iloc[0], 2))
        for mode in ('pure', 'e2e'):
            means = [x['mean'] for x in g[mode]]
            for k in ('mean', 'std', 'p50', 'p95', 'p99'):
                rec[f'{mode}_{k}'] = round(float(np.mean([x[k] for x in g[mode]])), 3)
            rec[f'{mode}_mean_range'] = f'{min(means):.3f}-{max(means):.3f}'
        rec['rss_delta_mb_max'] = round(max(x['rss_delta_mb'] for x in g['e2e']), 2)
        rows.append(rec)
    tab = pd.DataFrame(rows)
    pd.set_option('display.width', 250)
    print('\n=== RINGKASAN (rata-rata atas ulangan; *_mean_range = rentang mean antar-ulangan) ===')
    print(tab.to_string(index=False))
    return tab

## Setup: data, HuggingFace cache and `pcmef_fixed` (after the thread settings above)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys
shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_fixed.py', '/content/pcmef_fixed.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import pcmef_fixed as P   # must come after the first cell

import os
import pandas as pd

# HuggingFace models are cached on the LOCAL disk (the Drive cache can corrupt model files because
# the HF cache relies on symlinks, which Drive does not support reliably).
# Optional: add a Colab secret named HF_TOKEN and enable notebook access to avoid rate limits.
HF_CACHE = '/content/hf_cache'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (df['Speed'].round(2).astype(str) + '_' +
                           df['Scenario'].astype(str) + '_' +
                           (df['Time'] // 60).astype(int).astype(str))
    lo, hi = {15: (13, 17), 25: (23, 27)}[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

dfs15, _, _ = P.filter_feasible_episodes(load_and_filter(P15, 15), min_duration_sec=15)
dfs25, _, _ = P.filter_feasible_episodes(load_and_filter(P25, 25), min_duration_sec=15)

import os
OUT_NEXT = '/content/drive/MyDrive/Detector_Results/tahap_lanjutan'
os.makedirs(OUT_NEXT, exist_ok=True)
print('dfs15:', len(dfs15), 'rows | dfs25:', len(dfs25), 'rows')

## Pre-download the BERT models (retries on HF Hub errors and corrupted files)

Needed only for Part 2 (BERT); skip it if you only run Part 1. It retries on Hub errors (429/500) and re-downloads automatically if a cached file is corrupted. Models are cached on the local runtime disk (`/content/hf_cache`), so a new runtime downloads them again (~700 MB total).

If you ran an earlier version of this notebook that cached models on Drive, delete that folder first: `import shutil; shutil.rmtree('/content/drive/MyDrive/hf_cache', ignore_errors=True)`.

In [ ]:
import shutil
shutil.rmtree('/content/drive/MyDrive/hf_cache', ignore_errors=True)   # remove the old, possibly corrupted Drive cache

from transformers import (BertTokenizerFast, BertForSequenceClassification,
                          DistilBertTokenizerFast, DistilBertForSequenceClassification)
for _tok, _mdl, _name in [(BertTokenizerFast, BertForSequenceClassification, 'bert-base-uncased'),
                          (DistilBertTokenizerFast, DistilBertForSequenceClassification, 'distilbert-base-uncased')]:
    _hf_retry(_tok.from_pretrained, _name)
    _hf_retry(_mdl.from_pretrained, _name, num_labels=3)
    print('cached:', _name)

## Run the measurement in two parts

**Part 1 (fast, no BERT):** RF-Stat (source / modified), XGBoost-only, LSTM (source) and BiLSTM-15s, both regimes. Each pass takes milliseconds, so this part should finish in minutes (rough estimate, depends on the Colab CPU). Results go to `latency_suite.json`.

**Part 2 (slow, BERT only):** BERT-base at ~375 tokens on one CPU thread may need hundreds of ms per pass, so BERT (source) alone can take an hour or more per repeat (1,000 passes x 2 modes). The full BERT protocol (both regimes, 3 repeats) took about 7 hours. Results go to a SEPARATE file, `latency_suite_bert.json`, so Part 1 is never lost and BERT can be skipped or postponed. Options to shorten it:
- run 15 m/s only: `run_all({15: dfs15}, ...)` (the regime barely affects latency except for BERT modified);
- or pass `n_timed_bert=200` (about 14 min per repeat at 15 m/s) with a different output file name so earlier results are not overwritten. Either reduction MUST be stated in the manuscript.

Measured on a Colab 2-core CPU: BERT-base about 1.55 s and DistilBERT about 0.59 s per pass, i.e. about 80 min per repeat for 1,000 passes.

Note: the protocol says all configurations are measured in ONE session. If BERT is run later in a different session, say so in the manuscript (host block of each run is saved in its JSON).

Send back: the `HOST:` block, the `RINGKASAN` table, and the Keras `predict()` diagnostic row of LSTM (modified).

In [ ]:
# Part 1: all configurations except BERT
NON_BERT = [k for k in CONFIGS if not k.startswith('BERT')]
res = run_all({15: dfs15, 25: dfs25}, P, f'{OUT_NEXT}/latency_suite.json', configs=NON_BERT)

In [ ]:
# Part 2 (slow, optional): BERT only, separate output file.
# Full protocol (both regimes, 3 repeats, 1,000 passes) took about 7 hours on a Colab 2-core CPU
# (about 79 min per repeat at 15 m/s and about 62 min at 25 m/s).
BERT_ONLY = [k for k in CONFIGS if k.startswith('BERT')]
res_bert = run_all({15: dfs15, 25: dfs25}, P, f'{OUT_NEXT}/latency_suite_bert.json', configs=BERT_ONLY)

# Shorter variants (state the reduction in the manuscript; use a different file name so the full result is not overwritten):
# res_bert200 = run_all({15: dfs15}, P, f'{OUT_NEXT}/latency_suite_bert_200.json', configs=BERT_ONLY, repeats=3, n_timed_bert=200)

## Re-print the summary from the saved JSON (optional)

In [ ]:
summarize(json.load(open(f'{OUT_NEXT}/latency_suite.json')))        # Part 1
# summarize(json.load(open(f'{OUT_NEXT}/latency_suite_bert.json')))   # Part 2